# Loading and Accessing Data

This notebook demonstrates how to load datasets from manifests or archives and access different types of protein data. We'll explore the Pythonic API for working with sequences, structures, assays, and MSAs.

## Loading Datasets

There are two main ways to load a PG2 dataset:
1. From a manifest file (TOML)
2. From a dataset archive (ZIP)

Let's start by importing the necessary modules:

In [1]:
from pathlib import Path
from pg2_dataset import Dataset, Manifest

# Set up paths
manifest_path = Path("../example_data/neime_2019.toml")

### Method 1: Loading from Manifest

In [2]:
# Load manifest first
try:
    manifest = Manifest.from_path(manifest_path)
    print(f"Loaded manifest: {manifest.name}")
    print(f"Version: {manifest.version}")
    
    # Create dataset from manifest
    dataset = Dataset.from_manifest(manifest)
    print(f"\nDataset created successfully!")
    print(f"Dataset name: {dataset.name}")
    
except Exception as e:
    print(f"Error loading from manifest: {e}")
    print("This might be due to missing data files or incorrect paths.")

Loaded manifest: NEIME_2019
Version: 1.0.0

Dataset created successfully!
Dataset name: NEIME_2019


### Method 2: Loading from Archive

If you have a dataset archive (created in the previous notebook), you can load it directly:

In [3]:
# Look for existing archives
archive_path = "../example_data/NEIME_2019.zip"

try:
    dataset = Dataset.from_path(archive_path)
except Exception as e:
    print(f"Error loading from archive: {e}")
    print(f"Did you create an archive in the previous tutorial?")

## Exploring Dataset Structure

Let's examine what's in our dataset:

In [4]:
if 'dataset' in locals():
    print(f"Dataset: {dataset.name}")
    print(f"Description: {dataset.description}")
    print("\nDataset contents:")
    print(f"  - Sequences: {len(dataset.sequences)}")
    print(f"  - Structures: {len(dataset.structures)}")
    print(f"  - MSAs: {len(dataset.msas)}")
    print(f"  - Assays: {len(dataset.assays)}")
    print(f"  - Assay conditions: {len(dataset.assay_conditions)}")
else:
    print("Dataset not loaded. Please check the previous cells.")

Dataset: NEIME_2019
Description: NEIME enzyme dataset from Kennouche et al. 2019 with DMS scores

Dataset contents:
  - Sequences: 1
  - Structures: 1
  - MSAs: 1
  - Assays: 1
  - Assay conditions: 2


## Accessing Assays

In [5]:
# Access the assays
assays = dataset.assays

# Extract an specific assay
my_assay = assays[0]

# We can get a summary of the data encoded in this assay:
for field in my_assay.__class__.model_fields:
    print(f"Found a field for {field}")
    print(f"Pythonic description of the field information:")
    print(f"{my_assay.__class__.model_fields[field]}")
    print(f"------------")

Found a field for name
Pythonic description of the field information:
annotation=str required=True description='The name of the assay.'
------------
Found a field for records
Pythonic description of the field information:
annotation=list[tuple[str, Union[int, float, bool, str]]] required=True description='The records of the assay, pairs of Sequence and target values.'
------------
Found a field for conditions
Pythonic description of the field information:
annotation=dict[str, Union[int, float, bool, str]] required=False default_factory=dict description='The conditions of the assay, defined in the manifest.'
------------
Found a field for description
Pythonic description of the field information:
annotation=Union[str, NoneType] required=False default=None description='The description of the assay.'
------------
Found a field for sequence_feature_name
Pythonic description of the field information:
annotation=str required=False default='sequence' description='The sequence feature name in 

In [6]:
my_assay.__class__.model_fields['name']

FieldInfo(annotation=str, required=True, description='The name of the assay.')

In [7]:
# Access specific attributes such as name
name = my_assay.name
print(f"Assay name: {name}")

# Or extract the records
records = my_assay.records
print(f"{name} contains {len(records)} records")
print(f"record 1: {records[0][0][:20]}... with value {records[0][1]}")

Assay name: Assay1
Assay1 contains 922 records
record 1: ITLIELMIVIAIVGILAAVA... with value -3.5980000000000003


In [8]:
# Easy to obtain the sequences and targets for your ML application:
sequences = [record[0] for record in records]
values = [record[1] for record in records]

print([seq[0:20] + '..' for seq in sequences[0:5]])
print(values[0:5])


['ITLIELMIVIAIVGILAAVA..', 'LTLIELMIVIAIVGILAAVA..', 'YTLIELMIVIAIVGILAAVA..', 'VTLIELMIVIAIVGILAAVA..', 'STLIELMIVIAIVGILAAVA..']
[-3.5980000000000003, -0.6779999999999999, -2.373, 1.2990000000000002, -0.127]


## Accessing Assay Conditions

Assay conditions describe the experimental setup:

In [9]:
print(f"Number of assay conditions: {len(dataset.assay_conditions)}")
    
for i, condition in enumerate(dataset.assay_conditions):
    print(f"\nCondition {i+1}:")
    print(f"  - Name: {condition.name}")
    print(f"  - Description: {condition.description}")
    print(f"  - Unit: {condition.unit}")
    print(f"  - Value: {condition.value}")

Number of assay conditions: 2

Condition 1:
  - Name: temperature
  - Description: Reaction temperature
  - Unit: °C
  - Value: 37

Condition 2:
  - Name: pH
  - Description: Buffer pH
  - Unit: pH
  - Value: 7.4


## Accessing Structures

In [10]:
# Access the structures
structures = dataset.structures

# Obtain a specific structure:
my_structure = structures[0]

# We can get a summary of the metadata encoded in this assay:
for field in my_structure.__class__.model_fields:
    print(f"Found a field for {field}")
    print(f"Pythonic description of the field information:")
    print(f"{my_structure.__class__.model_fields[field]}")
    print(f"------------")

Found a field for name
Pythonic description of the field information:
annotation=str required=True description='The name of the protein structure.'
------------
Found a field for value
Pythonic description of the field information:
annotation=Structure required=True description='The value of the protein structure, typically a file path or binary data.'
------------
Found a field for description
Pythonic description of the field information:
annotation=Union[str, NoneType] required=False default=None description='The description of the protein structure.'
------------
Found a field for metadata
Pythonic description of the field information:
annotation=dict[str, str] required=False default_factory=dict description='Additional metadata for the protein structure.'
------------


When you access the structure, we return a biopython structure object. Biopython follows the so-called SMCRA (Structure/Model/Chain/Residue/Atom) architecture. Allowing you to access atoms from residues, residues from chains, chains from models and models from structures.

The following methods can be used to extract specific atom or residue data. See https://biopython.org/wiki/The_Biopython_Structural_Bioinformatics_FAQ for more information. The page contains helpful tips for accessing information from the structure object.

```python
atom.get_name()  # atom name (spaces stripped, e.g. 'CA')
atom.get_id()  # id (equals atom name)
atom.get_coord()  # atomic coordinates
atom.get_vector()  # atomic coordinates as Vector object
atom.get_bfactor()  # isotropic B factor
atom.get_occupancy()  # occupancy
atom.get_altloc()  # alternative location specifier
atom.get_sigatm()  # std. dev. of atomic parameters
atom.get_siguij()  # std. dev. of anisotropic B factor
atom.get_anisou()  # anisotropic B factor
atom.get_fullname()  # atom name (with spaces, e.g. '.CA.')
```

```python
residue.get_resname()  # return the residue name (eg. 'GLY')
residue.is_disordered()  # 1 if the residue has disordered atoms
residue.get_segid()  # return the SEGID
residue.has_id(name)  # test if a residue has a certain atom
```

In [11]:
biopython_structure = my_structure.value

def print_model_info(structure):
    """Prints out simple information describing the 

    Args:
        structure: biopython structure
    """
    for model in structure:
        print(model)
        for chain in model:
            print(chain)
            for residue in chain:
                print(residue.get_resname())
                for atom in residue:
                    print(atom.get_name())
                    return
                
print_model_info(biopython_structure)


<Model id=0>
<Chain id=A>
PHE
N


## Accessing MSAs (Multiple Sequence Alignments)

MSAs provide evolutionary information through aligned sequences. Similarly to structures we return the biopython object to access the msa data.

In [12]:
# Access the MSA data
msas = dataset.msas

my_msa = msas[0]
print(my_msa)

name='A0A1I9GEU1_NEIME' value=<<class 'Bio.Align.MultipleSeqAlignment'> instance (5553 records of length 161) at 106883560> description='Generated by Software X'


In [13]:
# A biopython MSA object is a list of sequence records:

biopython_msa = my_msa.value
print(biopython_msa)

Alignment with 5553 rows and 161 columns
FTLIELMIVIAIVGILAAVALPAYQDYTARAQVSEAILLAEGQK...sas A0A1I9GEU1_NEIME/1-161
----------------------------ARAQVSEAILLAEGQK...sa. UniRef100_UPI0018A25760/3-135
----------------------------ARAQVSEAILLAEGQK...... UniRef100_UPI00145ECBA7/3-130
FTLIELMIVIAIVGILAAVALPAYQDYTARAQVSEAILLAEGQK...... UniRef100_Q00045/8-162
----------------------------ARAQVSEAILLAEGQK...t.. UniRef100_A0A7U1LX12/3-136
----------------------------ARAQVSEAILLAEGQK...... UniRef100_UPI000BA48856/3-130
---------------------------------DEAILLAEGQK...t.. UniRef100_UPI0005E9AABB/13-136
---------------------------------DEAILLADGQK...s.. UniRef100_UPI0013E0C905/9-133
------------------------------AQVSEAILLAEGQK...sa. UniRef100_UPI0018C26155/1-133
---------------------------------DEAILLAEGQK...sa. UniRef100_D6HB30/9-137
FTLIELMIVIAIVGILAAVALPAYQDYTARAQVSEAILLAEGQK...d.. UniRef100_B0FXJ6/2-160
FTLIELMIVIAIVGILAAVALPAYQDYTARAQVSEAILLAEGQK...t.. UniRef100_UPI0005E0CA5C/8-166
FTLIELMIVIAIVGILA

In [14]:
# Accessing individual SeqRecords in the MSA:
print(biopython_msa[0])

# or
biopython_msa[0]

ID: A0A1I9GEU1_NEIME/1-161
Name: A0A1I9GEU1_NEIME/1-161
Description: A0A1I9GEU1_NEIME/1-161
Number of features: 0
Seq('FTLIELMIVIAIVGILAAVALPAYQDYTARAQVSEAILLAEGQKSAVTEYYLNH...sas')


SeqRecord(seq=Seq('FTLIELMIVIAIVGILAAVALPAYQDYTARAQVSEAILLAEGQKSAVTEYYLNH...sas'), id='A0A1I9GEU1_NEIME/1-161', name='A0A1I9GEU1_NEIME/1-161', description='A0A1I9GEU1_NEIME/1-161', dbxrefs=[])

In [15]:
# Allows us to obtain the SeqRecord:

biopython_msa[0].seq

Seq('FTLIELMIVIAIVGILAAVALPAYQDYTARAQVSEAILLAEGQKSAVTEYYLNH...sas')

## Accessing Sequences
We also record the reference sequence(s) of the dataset in the `[[ sequence ]]` section. This is helpful for engineering compared to a `wild-type` or `starting_sequence` and `engineered_sequence` from previous campaigns.

**It is important to note that the sequences in dataset.sequences are your reference sequences. The mutated sequences belong to an assay are what you most likely use for your ML model. See 

In [16]:
# Access the list of reference sequences
sequences = dataset.sequences

# Obtain a specific sequence
my_sequence = sequences[0]

print(f"Sequence name: {my_sequence.name}")
print(f"Sequence description: {my_sequence.description}")
print(f"Sequence type: {my_sequence.type}")
print(f"Sequence: {my_sequence.value[0:20]}....")



Sequence name: tr|A0A1I9GEU1|A0A1I9GEU1_NEIME
Sequence description: tr|A0A1I9GEU1|A0A1I9GEU1_NEIME Pilin OS=Neisseria meningitidis OX=487 PE=1 SV=1
Sequence type: SequenceType.WILD_TYPE
Sequence: FTLIELMIVIAIVGILAAVA....


## Summary

In this notebook, we've learned how to:

1. **Load datasets** from manifests and archives
2. **Access different data types**: sequences, structures, MSAs, and assays
3. **Access metadata** and conditions

The PG2 Dataset package provides a powerful and flexible way to work with protein data in machine learning workflows. The standardized API makes it easy to switch between different datasets while maintaining consistent code structure.

## Next Steps

Now you're ready to:
- Use PG2 Dataset in your own ML projects
- Share standardized datasets with collaborators
- Take a look at `04_Data_Access_for_ML` to get started connecting the data to your machine learning models.
